# Modelado para prediccion de variables

- Construye el dataset diario desde `sentiment_daily.parquet` y el CSV de precios.
- Evalúa 2 objetivos (targets):
  1) `target_parkvol_tomorrow` (volatilidad Parkinson de mañana)
  2) `target_absret_tomorrow` (|log-return| de mañana)
- Incluye dos features extra para capturar cambios y “intensidad” de noticias:
  - `sent_delta` = cambio del sentimiento vs el día anterior
  - `log_n_rows` = log(1 + nº de titulares del día)

Al final se guardan CSVs con los resultados (predicciones y métricas) en la carpeta `outputs_modelado/`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, RidgeCV, LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score, precision_score, recall_score
)

# Variables de configuracion
DATA_DIR = Path("data")

# Salida del notebook de análisis
SENTIMENT_DAILY_PATH = DATA_DIR / f"sentiment/sentiment_daily_finbert.parquet"
SENTIMENT_ROWS_PATH  = DATA_DIR / f"sentiment/sentiment_rows_finbert.parquet"

PRICE_CSV_PATH = Path("Datos históricos del Bitcoin.csv")  # CSV de precios

#Cache del dataset ya mergeado para no recalcular cada vez
DATASET_CACHE_CSV = Path("dataset_modelado.csv")
USE_CACHED_DATASET = True  # poner a True para leer DATASET_CACHE_CSV si existe

# Carpeta de salida
OUT_DIR = Path("outputs_modelado")
OUT_DIR.mkdir(exist_ok=True)

# Parametros de modelado
MIN_TRAIN_SIZE = 60 #Minimo de noticias a usar en entrenamiento
ROLL_WINDOWS = (7, 14) #Baselines rolling mean
USE_EXTENDED_FEATURES = True # Añade mas variables: lags/rollings/z-scores
HIGH_Q = 0.90 # Umbral para clasificación "alta volatilidad" (top 10%)

print("SENTIMENT_DAILY_PATH:", SENTIMENT_DAILY_PATH.resolve(), "| exists:", SENTIMENT_DAILY_PATH.exists())
print("PRICE_CSV_PATH:      ", PRICE_CSV_PATH.resolve(),       "| exists:", PRICE_CSV_PATH.exists())
print("DATASET_CACHE_CSV:   ", DATASET_CACHE_CSV.resolve(),    "| exists:", DATASET_CACHE_CSV.exists())
print("OUT_DIR:             ", OUT_DIR.resolve())


In [ ]:
# Construir dataset diario con sentiment_daily.parquet + csv precios
from typing import Optional

def parse_es_number(s):
    """Convierte números tipo '43.210,12' a float."""
    if pd.isna(s):
        return np.nan
    s = str(s).strip().replace(".", "").replace(",", ".")
    try:
        return float(s)
    except:
        return np.nan

def parse_volume(s):
    """Convierte '1,23M' o '456K' a float."""
    if pd.isna(s):
        return np.nan
    s = str(s).strip().upper().replace(" ", "")
    mult = 1.0
    if s.endswith("K"):
        mult, s = 1_000.0, s[:-1]
    elif s.endswith("M"):
        mult, s = 1_000_000.0, s[:-1]
    s = s.replace(".", "").replace(",", ".")
    try:
        return float(s) * mult
    except:
        return np.nan

def load_sentiment_daily_all(path_daily: Path) -> pd.DataFrame:
    """Carga sentiment_daily y devuelve el nivel 'ALL'.

    Dataset fijo con columnas: date (YYYY-MM-DD), source, n_rows, sentiment_mean, sentiment_median
    """
    df_s = pd.read_parquet(path_daily)

    # Nos quedamos con ALL
    df_all = df_s[df_s["source"].astype(str).str.upper() == "ALL"].copy()

    # Asegurar formato YYYY-MM-DD sin hora
    df_all["date"] = pd.to_datetime(df_all["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    df_all = df_all.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

    # Asegurar tipos
    df_all["n_rows"] = pd.to_numeric(df_all["n_rows"], errors="coerce").fillna(0).astype(int)
    for c in ["sentiment_mean", "sentiment_median"]:
        df_all[c] = pd.to_numeric(df_all[c], errors="coerce").fillna(0.0)

    return df_all

def load_price_csv(path_price: Path) -> pd.DataFrame:
    df_p = pd.read_csv(path_price)

    #1. Columna de fecha -> 'date' (YYYY-MM-DD)
    col_date = "Fecha" if "Fecha" in df_p.columns else ("Date" if "Date" in df_p.columns else None)
    if col_date is None:
        raise ValueError("No encuentro la columna de fecha (Fecha/Date) en el CSV de precios.")

    df_p["date"] = pd.to_datetime(df_p[col_date], errors="coerce", dayfirst=True).dt.strftime("%Y-%m-%d")
    df_p = df_p.dropna(subset=["date"]).copy()

    # Eliminar la columna original de fecha para que no aparezca en el CSV final
    if col_date in df_p.columns:
        df_p = df_p.drop(columns=[col_date])

    #2. Columnas de precios
    col_close = "Último"   if "Último"   in df_p.columns else ("Close"   if "Close"   in df_p.columns else None)
    col_open  = "Apertura" if "Apertura" in df_p.columns else ("Open"    if "Open"    in df_p.columns else None)
    col_high  = "Máximo"   if "Máximo"   in df_p.columns else ("High"    if "High"    in df_p.columns else None)
    col_low   = "Mínimo"   if "Mínimo"   in df_p.columns else ("Low"     if "Low"     in df_p.columns else None)
    col_vol   = "Vol."     if "Vol."     in df_p.columns else ("Volume"  if "Volume"  in df_p.columns else None)

    if col_close is None or col_high is None or col_low is None:
        raise ValueError("No encuentro columnas de precio (close/high/low). Revisar el CSV de precios.")

    df_p["close"]  = df_p[col_close].map(parse_es_number) if col_close in df_p.columns else np.nan
    df_p["open"]   = df_p[col_open].map(parse_es_number)  if col_open  in df_p.columns else np.nan
    df_p["high"]   = df_p[col_high].map(parse_es_number)
    df_p["low"]    = df_p[col_low].map(parse_es_number)
    df_p["volume"] = df_p[col_vol].map(parse_volume)      if col_vol   in df_p.columns else np.nan

    # Ordenar ascendente por fecha
    df_p = df_p.sort_values("date").reset_index(drop=True)

    return df_p

def build_dataset() -> pd.DataFrame:
    # 1. Sentimiento diario (ALL)
    df_s_all = load_sentiment_daily_all(SENTIMENT_DAILY_PATH)

    # 2. Precios
    df_p = load_price_csv(PRICE_CSV_PATH)

    # 3. Merge por día (date: YYYY-MM-DD)
    df = pd.merge(df_p, df_s_all, on="date", how="left").sort_values("date").reset_index(drop=True)

    # Si un día no tiene noticias, ponemos 0 (importante para sent_delta y log_n_rows)
    df["n_rows"] = df["n_rows"].fillna(0).astype(int)
    for c in ["sentiment_mean", "sentiment_median"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

    # 4. Features/targets (basados en precios)
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
    df["abs_log_return"] = df["log_return"].abs()

    df["parkinson_var"] = (1.0 / (4.0 * np.log(2.0))) * (np.log(df["high"] / df["low"]) ** 2)
    df["parkinson_vol"] = np.sqrt(df["parkinson_var"])

    HORIZONS = [1, 3, 7]  # Dias futuros a promediar

    for h in HORIZONS:
        df[f"target_absret_next{h}d"] = (
            df["abs_log_return"]
            .shift(-1)
            .rolling(window=h, min_periods=h)
            .mean()
            .shift(-(h-1))
        )
        df[f"target_parkvol_next{h}d"] = (
            df["parkinson_vol"]
            .shift(-1)
            .rolling(window=h, min_periods=h)
            .mean()
            .shift(-(h-1))
        )

    return df

# Construir dataset
if USE_CACHED_DATASET and DATASET_CACHE_CSV.exists():
    print("Leyendo dataset creado:", DATASET_CACHE_CSV)
    df = pd.read_csv(DATASET_CACHE_CSV, dtype={"date": str})
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
else:
    print("Construyendo dataset desde sentiment_daily.parquet + precios...")
    df = build_dataset()
    df.to_csv(DATASET_CACHE_CSV, index=False)
    print("Dataset guardado en:", DATASET_CACHE_CSV.resolve())

df = df.sort_values("date").reset_index(drop=True)

print("Dataset cargado:", df.shape)
print("Rango de fechas:", df["date"].min(), "a", df["date"].max())

df.head()


## Preparación de features
A partir del dataset diario (sentimiento + precios) creamos features temporales que suelen ayudar en series financieras:

- `sent_delta`: cambio diario del sentimiento  
- `sent_roll7/14`: medias móviles del sentimiento  
- `sent_z30`: z-score del sentimiento vs su media móvil 30d
- `log_n_rows`: log(1 + nº noticias) y sus medias móviles  
- `parkvol_roll7/14`, `absret_roll7/14`: medias móviles de volatilidad/|ret|  
- `dow_sin/dow_cos`: estacionalidad semanal

Luego se hace walk-forward (evaluación temporal: se entrena con el histórico disponible y se predice día a día el siguiente punto, sin usar información futura) con baselines fuertes (persistencia, expanding mean, rolling mean 7/14) y modelos (Ridge/RidgeCV y HistGradientBoostingRegressor).


In [ ]:
df = df.sort_values("date").reset_index(drop=True)

# Asegura columnas numéricas (por si CSV guardó como texto)
num_cols = [
    "parkinson_vol","sentiment_mean","sentiment_median","n_rows",
    TARGET_PARK,TARGET_ABS,
    "abs_log_return","log_return","volume","close","open","high","low"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Features de sentimiento (usamos sentiment_mean como serie principal)
df["sent_delta"] = df["sentiment_mean"] - df["sentiment_mean"].shift(1)
df["sent_lag1"] = df["sentiment_mean"].shift(1)
df["sent_roll7"] = df["sentiment_mean"].rolling(7, min_periods=3).mean()
df["sent_roll14"] = df["sentiment_mean"].rolling(14, min_periods=5).mean()

roll30_mean = df["sentiment_mean"].rolling(30, min_periods=10).mean()
roll30_std = df["sentiment_mean"].rolling(30, min_periods=10).std()
df["sent_z30"] = (df["sentiment_mean"] - roll30_mean) / (roll30_std.replace(0, np.nan))

# Conteo de noticias
df["log_n_rows"] = np.log1p(df["n_rows"].clip(lower=0))
df["log_n_rows_roll7"] = df["log_n_rows"].rolling(7, min_periods=3).mean()
df["log_n_rows_roll14"] = df["log_n_rows"].rolling(14, min_periods=5).mean()

# Volatilidad/retornos como features
df["parkvol_lag1"] = df["parkinson_vol"].shift(1)
df["parkvol_roll7"] = df["parkinson_vol"].rolling(7, min_periods=3).mean()
df["parkvol_roll14"] = df["parkinson_vol"].rolling(14, min_periods=5).mean()

df["absret_lag1"] = df["abs_log_return"].shift(1)
df["absret_roll7"] = df["abs_log_return"].rolling(7, min_periods=3).mean()
df["absret_roll14"] = df["abs_log_return"].rolling(14, min_periods=5).mean()

# Estacionalidad semanal: para representar el día de la semana de forma cíclica (seno/coseno) 
# y capturar posibles patrones semanales sin introducir saltos artificiales entre domingo y lunes. 
dow = pd.to_datetime(df["date"], errors="coerce").dt.dayofweek.astype(float)
df["dow_sin"] = np.sin(2*np.pi*dow/7.0)
df["dow_cos"] = np.cos(2*np.pi*dow/7.0)

# Conjuntos de features (USE_EXTENDED_FEATURES)
FEATURES_BASE = ["parkinson_vol","sentiment_mean","sent_delta","log_n_rows"]

FEATURES_EXT = [
    # “núcleo”
    "parkinson_vol","abs_log_return",
    "sentiment_mean","sent_delta","sent_roll7","sent_roll14","sent_z30",
    "log_n_rows","log_n_rows_roll7","log_n_rows_roll14",
    # suavizados con lag
    "parkvol_lag1","parkvol_roll7","parkvol_roll14",
    "absret_lag1","absret_roll7","absret_roll14",
    # estacionalidad
    "dow_sin","dow_cos",
]

FEATURES = FEATURES_EXT if USE_EXTENDED_FEATURES else FEATURES_BASE

# Ablación: separar features de sentimiento vs mercado
FEATURES_FULL = FEATURES 

# Sentimiento = columnas que empiezan por "sent" y las de "log_n_rows"
SENTIMENT_FEATURES = [c for c in FEATURES_EXT if c.startswith("sent") or c.startswith("log_n_rows")]
FEATURES_MARKET = [c for c in FEATURES_EXT if c not in SENTIMENT_FEATURES]

# Determina el horizonte objetivo
HORIZON_DAYS = 7 

TARGET_PARK = f"target_parkvol_next{HORIZON_DAYS}d"
TARGET_ABS  = f"target_absret_next{HORIZON_DAYS}d"
TARGETS = [TARGET_PARK, TARGET_ABS]

# Quitamos filas sin features o sin targets
df_model = df.dropna(subset=FEATURES + TARGETS).copy()

print("Filas totales:", len(df))
print("Filas para modelar:", len(df_model))
print("Features usadas:", FEATURES)

# Guardar dataset final
df_model.to_csv(OUT_DIR / "dataset_modelado.csv", index=False)
display(df_model[["date"] + FEATURES + TARGETS].head(5))


## Walk-forward, baselines fuertes, clasificación “alta volatilidad”
1. Regresión (2 objetivos/targets):
- `target_parkvol_tomorrow` (volatilidad Parkinson de mañana)
- `target_absret_tomorrow` (|retorno log| de mañana)
- Modelos: `Ridge`, `RidgeCV` y `HistGradientBoostingRegressor`

2. Clasificación:
- Se determina si mañana es un día de volatilidad alta si está en el top 10% (percentil `HIGH_Q`) del histórico disponible en el entrenamiento.
- Métricas: ROC-AUC, PR-AUC, F1, precision/recall.


In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

def make_reg_models():
    """Modelos candidatos para regresion."""
    models = {
        "ridge_a1": Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0))]),
        "ridgecv": Pipeline([("scaler", StandardScaler()), ("ridgecv", RidgeCV(alphas=np.logspace(-3, 3, 13)))]),
        "hgb": HistGradientBoostingRegressor(
            max_depth=3, learning_rate=0.05, max_iter=300,
            l2_regularization=0.1, random_state=42
        ),
    }
    return models

def compute_reg_metrics(pred_df: pd.DataFrame, y_col: str, pred_cols: list[str]) -> pd.DataFrame:
    rows = []
    y = pred_df[y_col].astype(float).values
    for c in pred_cols:
        p = pred_df[c].astype(float).values
        rows.append({
            "predictor": c,
            "n": len(pred_df),
            "mae": mean_absolute_error(y, p),
            "rmse": rmse(y, p),
            "r2": r2_score(y, p),
        })
    out = pd.DataFrame(rows).sort_values("mae")
    # skill vs mejor baseline (menor MAE entre baselines)
    baseline_cols = [c for c in pred_cols if c.startswith("base_")]
    if baseline_cols:
        best_base = out[out["predictor"].isin(baseline_cols)].sort_values("mae").iloc[0]
        out["best_baseline"] = best_base["predictor"]
        out["skill_vs_bestbase_mae"] = (out["mae"] - best_base["mae"]) / best_base["mae"]
    return out

def walk_forward_regression(
    df_in: pd.DataFrame,
    target_col: str,
    features: list[str],
    min_train_size: int = 60,
    roll_windows: tuple[int,int] = (7,14),
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Devuelve:
    - pred_df: predicciones por fecha
    - metrics_df: metricas por predictor
    """
    df_in = df_in.sort_values("date").reset_index(drop=True)

    preds = []
    models = make_reg_models()

    for i in range(min_train_size, len(df_in)):
        train = df_in.iloc[:i].copy()
        test = df_in.iloc[i:i+1].copy()

        # y_true del test
        y_true = float(test[target_col].values[0])
        if np.isnan(y_true):
            continue

        if "parkvol" in target_col:
            base_col = "parkinson_vol"
        elif "absret" in target_col:
            base_col = "abs_log_return"
        else:
            base_col = target_col  # fallback

        # historico hasta hoy (incluye el valor del dia test)
        hist = pd.concat([train[base_col], test[base_col]]).astype(float).dropna().values
        if len(hist) < min_train_size:
            continue

        base_persist = float(hist[-1])  # valor observado hoy
        base_expand = float(np.mean(hist))
        w1, w2 = roll_windows
        base_roll7  = float(np.mean(hist[-w1:])) if len(hist) >= w1 else base_expand
        base_roll14 = float(np.mean(hist[-w2:])) if len(hist) >= w2 else base_expand

        row = {
            "date": test["date"].values[0],
            "y_true": y_true,
            "base_persist": base_persist,
            "base_expanding_mean": base_expand,
            f"base_roll{w1}": base_roll7,
            f"base_roll{w2}": base_roll14,
        }

        # Entrenar modelos (si hay NaN, saltamos)
        X_train = train[features].values
        y_train = train[target_col].values
        X_test = test[features].values

        if np.isnan(X_train).any() or np.isnan(y_train).any() or np.isnan(X_test).any():
            continue

        for name, est in models.items():
            est_i = clone(est)
            est_i.fit(X_train, y_train)
            row[name] = float(est_i.predict(X_test)[0])

        preds.append(row)

    pred_df = pd.DataFrame(preds)
    pred_cols = [c for c in pred_df.columns if c not in ["date","y_true"]]
    metrics_df = compute_reg_metrics(pred_df, "y_true", pred_cols)
    return pred_df, metrics_df

# Clasificación “alta volatilidad”
def make_clf_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000))
    ])

def walk_forward_classification(
    df_in: pd.DataFrame,
    target_col: str,
    features: list[str],
    high_q: float = 0.90,
    min_train_size: int = 60,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Clasifica si el target esta en top-q."""
    df_in = df_in.sort_values("date").reset_index(drop=True)

    rows = []
    model = make_clf_model()

    for i in range(min_train_size, len(df_in)):
        train = df_in.iloc[:i].copy()
        test = df_in.iloc[i:i+1].copy()

        y_train = train[target_col].astype(float).values
        y_train = y_train[~np.isnan(y_train)]
        if len(y_train) < min_train_size:
            continue

        thr = float(np.quantile(y_train, high_q))
        y_true = float(test[target_col].values[0])
        if np.isnan(y_true):
            continue

        y_true_bin = int(y_true >= thr)

        X_train = train[features].values
        y_train_bin = (train[target_col].astype(float).values >= thr).astype(int)
        X_test = test[features].values

        if np.isnan(X_train).any() or np.isnan(X_test).any() or np.isnan(y_train_bin).any():
            continue

        m_i = clone(model)
        m_i.fit(X_train, y_train_bin)
        proba = float(m_i.predict_proba(X_test)[0,1])
        pred_bin = int(proba >= 0.5)

        rows.append({
            "date": test["date"].values[0],
            "thr": thr,
            "y_true": y_true,
            "y_true_high": y_true_bin,
            "proba_high": proba,
            "pred_high_0p5": pred_bin,
            "train_pos_rate": float(y_train_bin.mean()),
        })

    pred_df = pd.DataFrame(rows)
    if len(pred_df) == 0:
        return pred_df, pd.DataFrame()

    y = pred_df["y_true_high"].values
    p = pred_df["proba_high"].values

    metrics = pd.DataFrame([{
        "n": len(pred_df),
        "roc_auc": roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
        "pr_auc": average_precision_score(y, p) if len(np.unique(y)) > 1 else np.nan,
        "accuracy": accuracy_score(y, pred_df["pred_high_0p5"].values),
        "f1": f1_score(y, pred_df["pred_high_0p5"].values, zero_division=0),
        "precision": precision_score(y, pred_df["pred_high_0p5"].values, zero_division=0),
        "recall": recall_score(y, pred_df["pred_high_0p5"].values, zero_division=0),
        "high_q": high_q,
    }])
    return pred_df, metrics


In [ ]:

# Ejecutar regresión walk-forward para los 2 targets
# Ejecutar regresion walk-forward (con ablacion/eliminacion de variables) obteniendo dos versiones:
# 1. FULL: mercado + sentimiento
# 2. MARKET_ONLY: solo mercado

import pandas as pd

def to_df(x):
    """Convierte x a DataFrame si viene como lista de dicts, si ya es DF, lo copia."""
    if isinstance(x, pd.DataFrame):
        return x.copy()
    if isinstance(x, list):
        return pd.DataFrame(x)
    raise TypeError(f"Tipo no soportado para metrics: {type(x)}")

# FULL: mercado + sentimiento
pred_park_full, metrics_park_full = walk_forward_regression(
    df_model,
    target_col=TARGET_PARK,
    features=FEATURES_FULL,
    min_train_size=MIN_TRAIN_SIZE,
    roll_windows=ROLL_WINDOWS,
)
pred_abs_full, metrics_abs_full = walk_forward_regression(
    df_model,
    target_col=TARGET_ABS,
    features=FEATURES_FULL,
    min_train_size=MIN_TRAIN_SIZE,
    roll_windows=ROLL_WINDOWS,
)

# MARKET_ONLY: solo mercado (sin sentimiento)
pred_park_mkt, metrics_park_mkt = walk_forward_regression(
    df_model,
    target_col=TARGET_PARK,
    features=FEATURES_MARKET,
    min_train_size=MIN_TRAIN_SIZE,
    roll_windows=ROLL_WINDOWS,
)
pred_abs_mkt, metrics_abs_mkt = walk_forward_regression(
    df_model,
    target_col=TARGET_ABS,
    features=FEATURES_MARKET,
    min_train_size=MIN_TRAIN_SIZE,
    roll_windows=ROLL_WINDOWS,
)

# Convertir métricas a DataFrame y etiquetar
m_park_full = to_df(metrics_park_full)
m_abs_full  = to_df(metrics_abs_full)
m_park_mkt  = to_df(metrics_park_mkt)
m_abs_mkt   = to_df(metrics_abs_mkt)

m_park_full["target"] = TARGET_PARK
m_abs_full["target"]  = TARGET_ABS
m_park_mkt["target"]  = TARGET_PARK
m_abs_mkt["target"]   = TARGET_ABS

m_park_full["setting"] = "FULL"
m_abs_full["setting"]  = "FULL"
m_park_mkt["setting"]  = "MARKET_ONLY"
m_abs_mkt["setting"]   = "MARKET_ONLY"

df_metrics_ablation = pd.concat([m_park_full, m_abs_full, m_park_mkt, m_abs_mkt], ignore_index=True)
display(df_metrics_ablation.sort_values(["target", "mae"]))

# Guardar métricas
df_metrics_ablation.to_csv(OUT_DIR / "walkforward_metrics_ablation.csv", index=False)

# Predicciones
pred_park_full = pred_park_full.assign(target=TARGET_PARK, setting="FULL")
pred_abs_full  = pred_abs_full.assign(target=TARGET_ABS,  setting="FULL")
pred_park_mkt  = pred_park_mkt.assign(target=TARGET_PARK, setting="MARKET_ONLY")
pred_abs_mkt   = pred_abs_mkt.assign(target=TARGET_ABS,  setting="MARKET_ONLY")

df_preds_ablation = pd.concat([pred_park_full, pred_abs_full, pred_park_mkt, pred_abs_mkt], ignore_index=True)
df_preds_ablation.to_csv(OUT_DIR / "walkforward_predictions_ablation.csv", index=False)

print("Resultados guardados en outputs_modelado:")
print(" - walkforward_metrics_ablation.csv")
print(" - walkforward_predictions_ablation.csv")


# Clasificación
clf_pred_park, clf_metrics_park = walk_forward_classification(
    df_model, target_col=TARGET_PARK,
    features=FEATURES, high_q=HIGH_Q,
    min_train_size=MIN_TRAIN_SIZE,
)
clf_pred_abs, clf_metrics_abs = walk_forward_classification(
    df_model, target_col=TARGET_ABS,
    features=FEATURES, high_q=HIGH_Q,
    min_train_size=MIN_TRAIN_SIZE,
)

if isinstance(clf_metrics_park, pd.DataFrame) and len(clf_metrics_park):
    clf_metrics_park["target"] = TARGET_PARK
if isinstance(clf_metrics_abs, pd.DataFrame) and len(clf_metrics_abs):
    clf_metrics_abs["target"] = TARGET_ABS

print("Clasificación HIGH (top-q) - PARKVOL")
display(clf_metrics_park)

print("Clasificación HIGH (top-q) - ABSRET")
display(clf_metrics_abs)

In [ ]:
# Guardar CSVs con resultados

preds_ablation = pd.concat(
    [pred_park_full, pred_abs_full, pred_park_mkt, pred_abs_mkt],
    ignore_index=True
)

preds_path = OUT_DIR / "walkforward_predictions_ablation.csv"
preds_ablation.to_csv(preds_path, index=False)


def to_df(x):
    if isinstance(x, pd.DataFrame):
        return x.copy()
    if isinstance(x, list):
        return pd.DataFrame(x)
    raise TypeError(f"Tipo no soportado para metrics: {type(x)}")

m_park_full = to_df(metrics_park_full).assign(target=TARGET_PARK, setting="FULL")
m_abs_full  = to_df(metrics_abs_full).assign(target=TARGET_ABS,  setting="FULL")
m_park_mkt  = to_df(metrics_park_mkt).assign(target=TARGET_PARK, setting="MARKET_ONLY")
m_abs_mkt   = to_df(metrics_abs_mkt).assign(target=TARGET_ABS,  setting="MARKET_ONLY")

metrics_ablation = pd.concat([m_park_full, m_abs_full, m_park_mkt, m_abs_mkt], ignore_index=True)

metrics_path = OUT_DIR / "walkforward_metrics_ablation.csv"
metrics_ablation.to_csv(metrics_path, index=False)

if "clf_pred_park" in globals() and "clf_pred_abs" in globals():
    clf_pred_all = pd.concat(
        [
            clf_pred_park.assign(target=TARGET_PARK),
            clf_pred_abs.assign(target=TARGET_ABS),
        ],
        ignore_index=True
    )
    clf_pred_path = OUT_DIR / "walkforward_classification_predictions.csv"
    clf_pred_all.to_csv(clf_pred_path, index=False)

if "clf_metrics_park" in globals() and "clf_metrics_abs" in globals():
    clf_metrics_all = pd.concat([clf_metrics_park, clf_metrics_abs], ignore_index=True)
    clf_metrics_path = OUT_DIR / "walkforward_classification_metrics.csv"
    clf_metrics_all.to_csv(clf_metrics_path, index=False)

print("Guardado:")
print("-", preds_path)
print("-", metrics_path)
if "clf_pred_path" in locals():
    print(" -", clf_pred_path)
if "clf_metrics_path" in locals():
    print(" -", clf_metrics_path)